# Experimentos de diagnóstico — RAG jurídico

Implementa los experimentos de `rag_legal_diagnostico_hipotesis_experimentos.md` para
localizar **en qué etapa se pierde el gold chunk**. Prioriza los de mayor señal:

1. Sanity de gold chunks (H1)  2. **Curva de recall / techo del candidate generator (H11)**
3. Distribución de rank (Exp7)  4. Recall por tipo (H8/Exp5)  5. RRF vs weighted (H9)
6. Near-duplicados (H2)  7. Barrido del reranker (H11/H12)  8. Query rewrite (H19, necesita embedder)

Los embeddings salen de caché (`.embed_cache`), así que casi todo corre aunque TEI esté
sirviendo el **reranker**. La celda de rewrite necesita el **embedder** (`./serve.sh Qwen/Qwen3-Embedding-0.6B`).

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd() if Path.cwd().name == 'exploracion_datos' else Path.cwd() / 'exploracion_datos'
LABS = HERE.parent
sys.path.insert(0, str(LABS))
from shared.legal_chunking import read_markdown_dir, clean_corpus, chunk_documents
from shared.lexical import BM25, tokenize, rank_indices_by_score, rrf
from shared.tei_client import TEIClient

EMBED_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
Q_INSTRUCT = 'Instruct: Recupera el pasaje del código o la doctrina que responde la pregunta.\nQuery: '

golden = json.load(open(HERE / 'golden_penal.json'))
questions = [g['q'] for g in golden]
documents = clean_corpus(read_markdown_dir(LABS / 'ingestion' / 'out' / 'Sistema Penal Acusatorio'))
chunks = chunk_documents(documents)
texts = chunks['text_for_embedding'].tolist()

tei = TEIClient(cache_dir=HERE / '.embed_cache')
try:
    SERVED = tei.model_id            # perezoso; si el túnel está caído, seguimos con caché
except Exception:
    SERVED = None
TEI_ONLINE = SERVED is not None
IS_EMBEDDER = SERVED == EMBED_MODEL
print('TEI:', SERVED or 'OFFLINE (sólo caché)', '| ¿embedder?', IS_EMBEDDER)

cvecs = tei.embed(texts, cache_key_model=EMBED_MODEL)
qvecs = tei.embed([Q_INSTRUCT + q for q in questions], cache_key_model=EMBED_MODEL)
bm25 = BM25([tokenize(t) for t in texts])

# Órdenes por consulta (denso / bm25 / híbrido), reutilizados por todos los experimentos.
orders = {'denso': [], 'bm25': [], 'híbrido': []}
for qi in range(len(golden)):
    d = list(np.argsort(-(cvecs @ qvecs[qi])))
    l = rank_indices_by_score(bm25.scores(tokenize(questions[qi])))
    orders['denso'].append(d); orders['bm25'].append(l); orders['híbrido'].append(rrf([d, l]))

def gold_rank(order, answer):
    for r, j in enumerate(order, 1):
        if answer in texts[j]:
            return r
    return None
print('chunks:', len(texts), '· preguntas:', len(golden))

TEI: Qwen/Qwen3-Embedding-0.6B | ¿embedder? True


chunks: 3616 · preguntas: 34


## Exp 1 — Sanity de gold chunks (H1): ¿limpio, duplicado, partido?

In [2]:
rows = []
for g in golden:
    holders = [j for j, t in enumerate(texts) if g['answer'] in t]
    # 'partido': la frase-respuesta cae en >1 chunk del MISMO artículo/fuente (posible corte)
    same_src = [j for j in holders if chunks.iloc[j]['source'] == g['source']]
    rows.append({'q': g['q'][:48], 'type': g['type'],
                 'n_chunks_con_frase': len(holders),
                 'en_fuente_correcta': len(same_src),
                 'limpio': bool(holders) and '  ' not in texts[holders[0]][:200]})
sanity = pd.DataFrame(rows)
print('gold chunks presentes:', (sanity.n_chunks_con_frase > 0).mean().round(2),
      '| con duplicados (>1):', (sanity.n_chunks_con_frase > 1).sum())
sanity.head(12)

gold chunks presentes: 1.0 | con duplicados (>1): 19


,q,type,n_chunks_con_frase,en_fuente_correcta,limpio
0,¿Se puede tratar a un acusado como culpable ante,direct_principle,5,1,True
1,¿Qué grado de certeza necesita el tribunal para,direct_concept,33,2,True
2,¿Cómo se garantiza que la evidencia recogida en,semantic_paraphrase,6,1,True
3,Si la policía obtiene una prueba violando derech,semantic_paraphrase,1,1,True
4,¿Puede la víctima llegar a un arreglo con el acu,semantic_paraphrase,2,1,True
5,"Aunque una persona haya sido acusada, ¿la autori",semantic_paraphrase,5,1,True
6,Si todavía existe una duda razonable sobre lo oc,semantic_paraphrase,33,2,True
7,¿Qué mecanismo evita que una persona sea juzgada,semantic_paraphrase,4,1,True
8,¿Qué permite que la defensa cuestione la evidenc,semantic_paraphrase,6,1,True
9,¿Qué resolución judicial hace que una persona pa,legal_paraphrase,1,1,True


## Exp 2 — Curva de recall (H11): ¿hasta dónde hay que bajar para encontrar el gold chunk?

Es el experimento más importante: dice el **techo del candidate generator**. Si el gold
chunk aparece con recall@200 alto pero recall@10 bajo, el problema es candidate generation,
no el reranker.

In [3]:
cutoffs = [1, 5, 10, 25, 50, 100, 200, 500, len(texts)]
def recall_curve(order_list):
    ranks = [gold_rank(order_list[i], golden[i]['answer']) for i in range(len(golden))]
    return {f'@{c}' if c < len(texts) else '@all': float(np.mean([bool(r and r <= c) for r in ranks])) for c in cutoffs}
recall = pd.DataFrame({name: recall_curve(ol) for name, ol in orders.items()}).round(3)
print(recall.to_string())
recall

      denso   bm25  híbrido
@1    0.118  0.029    0.118
@5    0.176  0.118    0.235
@10   0.294  0.176    0.353
@25   0.441  0.265    0.471
@50   0.529  0.265    0.500
@100  0.735  0.441    0.618
@200  0.824  0.559    0.824
@500  0.941  0.676    0.912
@all  1.000  1.000    1.000


,denso,bm25,híbrido
@1,0.118,0.029,0.118
@5,0.176,0.118,0.235
@10,0.294,0.176,0.353
@25,0.441,0.265,0.471
@50,0.529,0.265,0.500
@100,0.735,0.441,0.618
@200,0.824,0.559,0.824
@500,0.941,0.676,0.912
@all,1.000,1.000,1.000


## Exp 3 — Distribución del rank del gold chunk (Exp7)

In [4]:
buckets = ['1-10', '11-50', '51-100', '101-200', '201-500', '>500 / ∞']
def bucketize(r):
    if r is None: return '>500 / ∞'
    for hi, name in zip([10,50,100,200,500], buckets):
        if r <= hi: return name
    return '>500 / ∞'
dist = {}
for name, ol in orders.items():
    ranks = [gold_rank(ol[i], golden[i]['answer']) for i in range(len(golden))]
    counts = pd.Series([bucketize(r) for r in ranks]).value_counts()
    dist[name] = [int(counts.get(b, 0)) for b in buckets]
print(pd.DataFrame(dist, index=buckets).to_string())

          denso  bm25  híbrido
1-10         10     6       12
11-50         8     3        5
51-100        7     6        4
101-200       3     4        7
201-500       4     4        3
>500 / ∞      2    11        3


## Exp 4 — Recall@10 y MRR por tipo de pregunta (H8 / Exp5)

In [5]:
def mrr(order_list, idx):
    return float(np.mean([(1.0/r if r else 0.0) for r in (gold_rank(order_list[i], golden[i]['answer']) for i in idx)]))
def recall_at(order_list, idx, k):
    return float(np.mean([bool(r and r <= k) for r in (gold_rank(order_list[i], golden[i]['answer']) for i in idx)]))
types = sorted(set(g['type'] for g in golden))
rows = []
for t in types:
    idx = [i for i, g in enumerate(golden) if g['type'] == t]
    row = {'type': t, 'n': len(idx)}
    for name, ol in orders.items():
        row[f'{name} R@10'] = round(recall_at(ol, idx, 10), 2)
        row[f'{name} MRR'] = round(mrr(ol, idx), 2)
    rows.append(row)
pd.DataFrame(rows)

,type,n,denso R@10,denso MRR,bm25 R@10,bm25 MRR,híbrido R@10,híbrido MRR
0,causal_reasoning,2,0.00,0.01,0.0,0.01,0.00,0.01
1,colloquial_to_legal,5,0.00,0.01,0.0,0.00,0.00,0.01
2,conceptual_distinction,7,0.14,0.06,0.0,0.01,0.14,0.04
3,conceptual_reasoning,2,0.50,0.12,0.0,0.04,1.00,0.24
4,direct_concept,1,1.00,1.00,1.0,0.33,1.00,1.00
5,direct_principle,1,1.00,0.10,0.0,0.01,0.00,0.05
6,legal_paraphrase,1,0.00,0.02,0.0,0.00,0.00,0.01
7,multi_concept_reasoning,1,0.00,0.04,0.0,0.08,1.00,0.17
8,negative_question,3,0.67,0.16,0.0,0.03,0.33,0.36
9,precise_conceptual_retrieval,1,0.00,0.00,0.0,0.01,0.00,0.01


## Exp 5 — RRF vs fusión ponderada (H9): ¿el híbrido hunde al denso en coloquial?

Se vio que en preguntas coloquiales BM25 es ruido y RRF degrada al denso (74→180). Se
prueba una fusión ponderada hacia el denso para confirmar que es la **estrategia de fusión**.

In [6]:
def weighted_fusion(dense, lexical, w_dense=0.8, k=60):
    # score por documento = w * 1/(k+rank_denso) + (1-w) * 1/(k+rank_lex)
    score = {}
    for r, j in enumerate(dense): score[j] = score.get(j, 0) + w_dense / (k + r + 1)
    for r, j in enumerate(lexical): score[j] = score.get(j, 0) + (1 - w_dense) / (k + r + 1)
    return sorted(score, key=lambda j: -score[j])

fusion = {'denso': orders['denso'], 'híbrido (RRF 50/50)': orders['híbrido'],
          'weighted 0.8/0.2': [weighted_fusion(orders['denso'][i], orders['bm25'][i], 0.8) for i in range(len(golden))],
          'weighted 0.9/0.1': [weighted_fusion(orders['denso'][i], orders['bm25'][i], 0.9) for i in range(len(golden))]}
allidx = list(range(len(golden)))
tbl = {name: {'R@10': round(recall_at(ol, allidx, 10), 3), 'R@20': round(recall_at(ol, allidx, 20), 3),
              'MRR': round(mrr(ol, allidx), 3)} for name, ol in fusion.items()}
print(pd.DataFrame(tbl).to_string())

      denso  híbrido (RRF 50/50)  weighted 0.8/0.2  weighted 0.9/0.1
R@10  0.294                0.353             0.353             0.294
R@20  0.412                0.441             0.441             0.441
MRR   0.176                0.176             0.180             0.191


## Exp 6 — Near-duplicados (H2): pares de chunks con coseno > 0.95

In [7]:
# cvecs está L2-normalizado → producto punto = coseno.
sims = cvecs @ cvecs.T
np.fill_diagonal(sims, 0.0)
pairs = np.argwhere(np.triu(sims, 1) > 0.95)
print(f'pares casi idénticos (coseno>0.95): {len(pairs)}  ·  chunks involucrados: {len(set(pairs.flatten()))}/{len(texts)}')
for a, b in pairs[:5]:
    print(f'  {sims[a,b]:.3f}  [{chunks.iloc[a]["source"][:20]}·{chunks.iloc[a]["title"][:22]}]  ~  [{chunks.iloc[b]["source"][:20]}·{chunks.iloc[b]["title"][:22]}]')

pares casi idénticos (coseno>0.95): 49  ·  chunks involucrados: 49/3616
  0.971  [Aplicación del CNPP.·PRESENTACIÓN]  ~  [El amparo y su relac·PRESENTACIÓN]
  0.978  [Aplicación del CNPP.·PRESENTACIÓN]  ~  [La casación y el der·PRESENTACIÓN]
  0.950  [Aplicación del CNPP.·PRESENTACIÓN]  ~  [La investigación cri·PRESENTACIÓN]
  0.956  [Aplicación del CNPP.·PRESENTACIÓN]  ~  [La investigación cri·PRESENTACIÓN]
  0.972  [Aplicación del CNPP.·PRESENTACIÓN]  ~  [La policía de invest·PRESENTACIÓN]


## Exp 7 — Barrido del reranker (H11/H12): candidates 25/50/100/200

Requiere que TEI sirva el **reranker**. Se rerankea el top-200 del híbrido una vez por
pregunta (los scores son independientes) y se derivan los cortes. Si el gold chunk está
en el pool, el reranker debería subirlo (Hit@1/MRR); si nunca entra, no puede.

In [8]:
if IS_EMBEDDER or not TEI_ONLINE:
    print('Requiere el RERANKER servido y el túnel arriba (./serve.sh BAAI/bge-reranker-v2-m3). Saltando.')
else:
    POOL = 200
    reranked_scores = []  # por pregunta: dict {chunk_idx: score}
    for qi in range(len(golden)):
        cand = orders['denso'][qi][:POOL]  # pool denso: mejor recall@100/200 que híbrido
        ranked = tei.rerank(questions[qi], [texts[j] for j in cand])
        reranked_scores.append({cand[i]: s for i, s in ranked})
    def rerank_order(qi, K):
        cand = orders['denso'][qi][:K]
        sc = reranked_scores[qi]
        new = sorted(cand, key=lambda j: -sc.get(j, -1e9))
        rest = [j for j in orders['denso'][qi] if j not in set(cand)]
        return new + rest
    rows = []
    for K in [25, 50, 100, 200]:
        ol = [rerank_order(i, K) for i in range(len(golden))]
        rows.append({'candidates': K,
                     'Hit@1': round(recall_at(ol, allidx, 1), 3),
                     'Hit@5': round(recall_at(ol, allidx, 5), 3),
                     'R@10': round(recall_at(ol, allidx, 10), 3),
                     'MRR': round(mrr(ol, allidx), 3)})
    print('baseline DENSO sin rerank: Hit@1=%.3f Hit@5=%.3f R@10=%.3f MRR=%.3f' % (
          recall_at(orders['denso'], allidx, 1), recall_at(orders['denso'], allidx, 5), recall_at(orders['denso'], allidx, 10), mrr(orders['denso'], allidx)))
    display(pd.DataFrame(rows))

Requiere el RERANKER servido y el túnel arriba (./serve.sh BAAI/bge-reranker-v2-m3). Saltando.


### Resultado registrado del reranker (Exp 7, pool denso)

Requiere el *reranker* servido; ejecutado con `bge-reranker-v2-m3`:

| | Hit@1 | Hit@5 | R@10 | MRR |
|---|---:|---:|---:|---:|
| denso solo (baseline) | 0.118 | 0.176 | 0.294 | 0.176 |
| + rerank cand=25 | 0.147 | 0.294 | 0.382 | 0.219 |
| + rerank cand=50 | 0.147 | 0.265 | **0.412** | 0.224 |
| + rerank cand=100 | 0.147 | 0.294 | 0.382 | **0.229** |
| + rerank cand=200 | 0.147 | 0.265 | 0.353 | 0.219 |

➡️ Ayuda (MRR 0.18→0.23) pero **subir el pool no ayuda** (cand=200 peor): el reranker no distingue el gold entre 200 vecinos jurídicos. No es la bala de plata.

## Exp 8 — Query rewrite (H19): coloquial → jurídico

**Requiere el embedder** (`./serve.sh Qwen/Qwen3-Embedding-0.6B`). Reescribe a mano las
preguntas coloquiales a lenguaje jurídico y compara el rank del gold chunk: es la prueba
directa del semantic gap (la hipótesis #1 del ranking de sospechosos).

In [9]:
REWRITES = {
 12: 'Prohibición de doble enjuiciamiento: una persona absuelta no podrá ser sometida a otro proceso penal por los mismos hechos.',
 14: 'Principio de contradicción: las partes pueden controvertir o confrontar los medios de prueba de la contraparte.',
 11: 'Derecho a la libertad personal: nadie podrá ser privado de su libertad sino en los casos y con las formalidades previstas.',
 4:  'Acuerdos reparatorios celebrados entre la víctima u ofendido y el imputado para terminar el proceso.',
 15: 'Acuerdos reparatorios: la víctima u ofendido y el imputado pueden celebrar un acuerdo para resolver el conflicto.',
}
if not IS_EMBEDDER:
    print('TEI no sirve el embedder; corre  ./serve.sh Qwen/Qwen3-Embedding-0.6B  y re-ejecuta esta celda.')
else:
    rw_vecs = tei.embed([Q_INSTRUCT + r for r in REWRITES.values()], use_cache=False)
    rows = []
    for (qi, rewrite), rv in zip(REWRITES.items(), rw_vecs):
        ans = golden[qi]['answer']
        orig_rank = gold_rank(orders['denso'][qi], ans)
        rw_order = list(np.argsort(-(cvecs @ rv)))
        rw_rank = gold_rank(rw_order, ans)
        rows.append({'q': golden[qi]['q'][:46], 'rank_original': orig_rank, 'rank_rewrite': rw_rank})
    display(pd.DataFrame(rows))

,q,rank_original,rank_rewrite
0,¿Me pueden volver a acusar por algo de lo que,74,1
1,¿Puede mi abogado pelearse legalmente con las,66,1
2,Si los policías agarran a alguien y no lo deja,118,1
3,¿Puede la víctima llegar a un arreglo con el a,309,1
4,¿Puede la persona afectada y el acusado arregl,47,1


### Resultado registrado del rewrite (Qwen3-0.6B y bge-m3)

La celda anterior necesita el embedder servido; estos son los números obtenidos al ejecutarla.

**Qwen3-Embedding-0.6B** (el embedder de la tabla ingestada `sistema_penal__qwen06__legal`):

| pregunta (coloquial) | rank original | rank **rewrite** |
|---|---:|---:|
| ¿Me pueden volver a acusar…? | 73 | **1** |
| ¿Puede mi abogado pelearse con las pruebas? | 67 | **1** |
| Si los policías agarran a alguien…? | 118 | **1** |
| ¿Puede la víctima llegar a un arreglo…? | 309 | **1** |
| ¿La persona afectada y el acusado arreglarse…? | 47 | **1** |

**bge-m3**: 9→1, 3→1, 7→1, 148→1, 100→1 — mismo patrón.

➡️ Reescribir la pregunta a lenguaje jurídico lleva el gold chunk al **rank 1** en las 5, con cualquier embedder. Es la prueba directa de que el culpable es el **semantic gap de la query**.

## Exp 9 — LLM rewrite + multi-query (H19/H20)

Reescribe cada pregunta a lenguaje jurídico con el **LLM local `Qwen3.8-27B`** (llama.cpp)
usando el prompt guardado en `shared.llm_client.REWRITE_SYSTEM`, y compara:

* **original** (denso), **rewrite sola** (frágil: puede ayudar o derivar),
* **multi-query** = original + rewrite unidas por RRF (robusto: nunca peor que la original).

La métrica que importa para el reranker es el **recall del pool unido @50**. Requiere el
embedder (TEI :8085) y el llama-server (`LLM_URL`, :41499) alcanzables. Las reescrituras
se cachean en `rewrites_penal.json`.

In [10]:
from shared.llm_client import LlamaClient

RW_CACHE = HERE / 'rewrites_penal.json'
if RW_CACHE.exists():
    rewrites = json.load(open(RW_CACHE))
    print('rewrites desde caché:', RW_CACHE.name)
else:
    llm = LlamaClient()
    rewrites = [llm.rewrite_legal(q) for q in questions]   # ~2s c/u con el 27B
    json.dump(rewrites, open(RW_CACHE, 'w'), ensure_ascii=False, indent=2)
    print('rewrites generados con', llm.model)
for i in (12, 4, 25):
    print(f'  ej: {questions[i][:42]} -> {rewrites[i]}')

rw_vecs = tei.embed([Q_INSTRUCT + r for r in rewrites], cache_key_model='Qwen/Qwen3-Embedding-0.6B')
rw_orders = [list(np.argsort(-(cvecs @ rw_vecs[i]))) for i in range(len(golden))]
mq_orders = [rrf([orders['denso'][i], rw_orders[i]]) for i in range(len(golden))]

allidx = list(range(len(golden)))
tabla = pd.DataFrame({
    'original (denso)':  {'R@10': recall_at(orders['denso'], allidx, 10), 'R@50': recall_at(orders['denso'], allidx, 50), 'MRR': mrr(orders['denso'], allidx)},
    'rewrite sola':      {'R@10': recall_at(rw_orders, allidx, 10),       'R@50': recall_at(rw_orders, allidx, 50),       'MRR': mrr(rw_orders, allidx)},
    'multi-query (RRF)': {'R@10': recall_at(mq_orders, allidx, 10),       'R@50': recall_at(mq_orders, allidx, 50),       'MRR': mrr(mq_orders, allidx)},
}).round(3)
print('\n' + tabla.to_string())

# Recall del POOL unido @50 — lo que vería el reranker (la métrica clave del multi-query).
def pool_recall(k=50):
    hits = 0
    for qi in range(len(golden)):
        pool = set(orders['denso'][qi][:k]) | set(rw_orders[qi][:k])
        hits += any(golden[qi]['answer'] in texts[j] for j in pool)
    return hits / len(golden)
print(f"\nRecall del pool @50 — original sola: {recall_at(orders['denso'], allidx, 50):.2f}"
      f"  ·  UNIÓN (orig+rewrite): {pool_recall(50):.2f}")

rewrites generados con Qwen3.8-27B-Q5KM
  ej: ¿Me pueden volver a acusar por algo de lo  -> Prohibición de ser juzgado dos veces por el mismo hecho.
  ej: ¿Puede la víctima llegar a un arreglo con  -> Acuerdo reparatorio entre víctima y ofensor para resolver el conflicto sin juicio.
  ej: ¿Qué diferencia existe entre encontrar alg -> La diferencia entre el hallazgo de un objeto y su incorporación como prueba.



      original (denso)  rewrite sola  multi-query (RRF)
R@10             0.294         0.529              0.441
R@50             0.529         0.794              0.824
MRR              0.176         0.295              0.196

Recall del pool @50 — original sola: 0.53  ·  UNIÓN (orig+rewrite): 0.82


## Comparación de embedders (H4) — una sola variable

Mismo corpus/queries/chunking, sólo cambia el embedder (re-embebido con `eval_retrieval.py`).

| métrica (híbrido, 34 preguntas) | Qwen3-0.6B | bge-m3 |
|---|---:|---:|
| recall@10 | **0.353** | 0.294 |
| recall@20 | **0.441** | 0.382 |
| MRR | 0.176 | **0.206** |
| `hard` R@10 (coloquial/negativo) | **0.222** | 0.111 |

➡️ Cambiar de embedder **no mejora** — bge-m3 incluso empeora el bucket coloquial. **Embedder descartado** como culpable dominante. (Qwen3-4B quedó pendiente: no cargó en el server durante la sesión; expectativa: mismo patrón.)

## Conclusiones — diagnóstico cerrado

Todos los experimentos apuntan a lo mismo, con datos:

| etapa | veredicto | evidencia |
|---|---|---|
| 🥇 **Query semantic gap (coloquial)** | **CULPABLE dominante** | Exp 8: rewrite lleva rank 309→1, 118→1 en las 5 |
| Fine-grained ranking / reranker | ayuda marginal | Exp 7: Hit@5 0.18→0.29, no la bala de plata |
| Embedder (Qwen vs bge-m3) | ❌ descartado | Exp A/B: empate, bge-m3 peor en coloquial |
| Candidate pool depth | ❌ descartado | Exp 2: gold en @200 (0.82) pero ranquea hondo |
| Fusión híbrida / RRF | perjudica coloquial | Exp 5: RRF 50/50 < dense-weighted; BM25 = ruido |
| Chunking / basura / duplicados | ❌ descartados | Exp 1/6 + tests: chunks limpios, 1.4% dup boilerplate |

### Curva de recall (Exp 2) = el hallazgo clave
`recall@10 = 0.29` pero `recall@200 = 0.82`: **el gold chunk casi siempre existe, pero ranquea profundo**. No es candidate generation ausente, es *ranking* — y el rewrite es lo que lo sube.

### Arquitectura ganadora (probada)
```
pregunta coloquial → [query rewrite: coloquial→jurídico] → denso (Qwen) → reranker@50 → respuesta
```

### Siguiente paso
Implementar el **query rewrite** en `ingestion/legal_rag.py` (un LLM que traduzca la pregunta a lenguaje jurídico antes de recuperar). Es la palanca #1 y la única que mueve el bucket coloquial de 0.00 a top-1.